# 第30章 变分自编码器

## 习题30.1

&emsp;&emsp;设计一个由2层卷积神经网络编码器和2层卷积神经网络解码器组成的自编码器AE（使用第31章介绍的转置卷积）。

**解答：**

**解答思路：**  

1. 给出自动编码器的原理
2. 给出转置卷积的定义
3. 自编程实现2层卷积自动编码器

**解答步骤：**

**第1步：自动编码器的原理**

&emsp;&emsp;根据书中第27.2.2节的自动编码器的描述：

> &emsp;&emsp;自动编码器（auto encoder）是用于数据表示的无监督学习的一种神经网络。自动编码器由编码器网络和解码器网络组成。
> 最基本的情况下，编码器和解码器分别都是一层神经网络
> $$ z = F(x) = a(W_E x + b_E) \\
y = G(z) = a(W_D x + b_D) $$
> 其中，$W_E, W_D$是权重矩阵，$b_E, b_D$是偏置向量，$a(\cdot)$是激活函数。  
> &emsp;&emsp;学习时，目标函数是
> $$ L = \frac{1}{N} \sum_{i=1}^N L(x_i, y_i) = \frac{1}{N} \sum_{i = 1}^N L(x_i, G(F(x_i))) $$

**第2步： 转置卷积的定义**

&emsp;&emsp;根据书中第28.2.1节的转置卷积的定义：

> &emsp;&emsp;转置卷积（transposed convolution）也称为微步卷积（fractionally strided convolution）或反卷积（deconvolution），在图像生成网络、图像自动编码器等模型中广泛使用。卷积可以用于图像数据尺寸的缩小，而转置卷积可以用于图像数据尺寸的放大，又分别称为下采样或上采样。

&emsp;&emsp;根据书中第28章的本章概要的转置卷积的运算：

> 对于任意一个卷积运算，存在对应的线性变换的矩阵 $C$。针对转置矩阵 $C^T$，引入新的卷积运算，称为转置卷积。原始卷积和转置卷积是相互对应、互为反向的运算。原始卷积的卷积核是 $W$ 时，转置卷积的卷积核是 $\text{rot180}(W)$。卷积核和转置卷积核之间有 $\text{rot180}(\text{rot180}(W)) = W$ 成立。

**第3步：自编程实现2层卷积自动编码器**

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import tqdm
from matplotlib import pyplot as plt
from torch import optim
from torch.utils.data import DataLoader
from torchvision.datasets import mnist
from torchviz import make_dot

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        # 2层卷积神经网络编码器
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1),
            nn.ReLU()
        )
        # 2层卷积神经网络解码器
        self.decoder = nn.Sequential(
            # 使用转置卷积
            nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 使用MNIST数据集
train_set = mnist.MNIST('./data', transform=transforms.ToTensor(), train=True, download=True)
test_set = mnist.MNIST('./data', transform=transforms.ToTensor(), train=False, download=True)
train_dataloader = DataLoader(train_set, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_set, batch_size=8, shuffle=False)

In [ ]:
model = AutoEncoder().to(device)

# 设置损失函数
criterion = nn.MSELoss()
# 设置优化器
optimizer = optim.Adam(model.parameters(), lr=1e-2)

In [ ]:
# 显示模型结构
model

In [ ]:
# 模型训练
EPOCHES = 10
for epoch in range(EPOCHES):
    for img, _ in tqdm.tqdm(train_dataloader, ):
        optimizer.zero_grad()

        img = img.to(device)
        out = model(img)
        loss = criterion(out, img)
        loss.backward()

        optimizer.step()

In [ ]:
# 将生成图片和原始图片进行对比
for i, data in enumerate(test_dataloader):
    img, _ = data
    img = img.to(device)
    model = model.to(device)
    img_new = model(img).detach().cpu().numpy()
    img = img.cpu().numpy()
    plt.figure(figsize=(8, 2))
    for j in range(8):
        plt.subplot(2, 8, j + 1)
        plt.axis('off')
        plt.imshow(img_new[j].squeeze())
        plt.subplot(2, 8, 8 + j + 1)
        plt.axis('off')
        plt.imshow(img[j].squeeze())
    if i >= 2:
        break

In [ ]:
def save_model_structure(model, device):
    x = torch.randn(1, 1, 28, 28).requires_grad_(True).to(device)
    y = model(x)
    vise = make_dot(y, params=dict(list(model.named_parameters()) + [('x', x)]))
    vise.format = "png"
    vise.directory = "./data"
    vise.view("2-Layer-CNN-AutoEncoder-Struction", cleanup=True)

In [ ]:
# 保存模型架构图
save_model_structure(model, device)

## 习题30.2

&emsp;&emsp;证明当编码器和解码器都是线性函数时，主成分分析可以作为AE学习的方法。

**解答：**

**解答思路：**  

1. 给出主成分分析的定义
2. 给出自动编码器学习过程
3. 证明当编码器和解码器都是线性函数时，主成分分析可以作为自动编码器学习的方法

**解答步骤：**

**第1步：主成分分析的定义**

&emsp;&emsp;根据书中第16章的主成分分析的基本思想：

> &emsp;&emsp;主成分分析中，首先对给定数据进行规范化，使得数据每一变量的平均值为0，方差为1，之后对数据进行正交变换，原来由线性相关变量表示的数据通过正交变换变成由若干个线性无关的新变量表示的数据。新变量是可能的正交变换中变量的方差的和（信息保存）最大的，方差表示在新变量上信息的大小。将新变量依次称为第一主成分、第二主成分等。这就是主成分分析的基本思想。

&emsp;&emsp;根据书中第16.2.3节的算法16.1主成分分析算法：

> **算法16.1（主成分分析算法）**  
> 输入：$m \times n$样本矩阵$X$，其每一行元素的均值为零；  
> 输出：$k \times n$样本主成分矩阵$Y$。  
> 参数：主成分个数$k$  
> （1）构造新的$n \times m$矩阵
> $$
X' = \frac{1}{\sqrt{n - 1}} X^T
$$
> $X'$每一列的均值为零。  
> （2）对矩阵$X'$进行截断奇异值分解，得到
> $$
X' = U \Sigma V^T
$$
> 有$k$个奇异值、奇异向量。矩阵$V$的前$k$列构成$k$个样本主成分。  
> （3）求$k \times n$样本主成分矩阵
> $$
Y = V^T X
$$

**第2步：自动编码器学习过程**

&emsp;&emsp;根据书中第27章的本章概要的自动编码器的描述：

> &emsp;&emsp;自动编码器是用于数据表示的无监督学习的一种神经网络。自动编码器由编码器网络和解码器网络组成。学习时编码器将输入向量转换为中间表示向量，解码器再将中间表示向量转换为输出向量。编码器和解码器可以是
> $$
z = F(x) = a(W_E x + b_E) \\
y = G(z) = a(W_D z + b_D)
$$
> 学习的目标是尽量使输出向量和输入向量保持一致，或者说重建输入向量。认为学到的中间表示向量就是数据的表示。
> $$
L = \frac{1}{N} \sum_{i = 1}^N L(x_i, G(F(x_i)))
$$
> 学习的算法一般是梯度下降。自动编码器学习实际进行的是对数据的压缩。

**第3步：证明当编码器和解码器都是线性函数时，主成分分析可以作为自动编码器学习的方法**

&emsp;&emsp;假设给定样本矩阵$X = (x_1, x_2, \cdots, x_N)$，根据主成分分析算法，可以得到主成分为$K$个对应的单位特征向量$V$，样本主成分矩阵$Y$满足
$$
Y = V^T \cdot X
$$

这样，样本主成分矩阵$Y$可以作为样本矩阵$X$的低维表示，$V$维度是$N \times K$，其中$V = (v_1, v_2, \cdots, v_N)$

&emsp;&emsp;根据自动编码器中的编码阶段，编码器对数据进行压缩，当编码器是线性函数时，可用主成分矩阵$Y$表示编码器的输出，即
$$
z = F(x) = W_E x + b_E = V^T \cdot x
$$
其中$W_E = V^T, b_E = 0$ 

&emsp;&emsp;在解码阶段，解码器通过解压可以得到原始数据的近似，当解码器是线性函数时，可用单位特征向量$V$表示$W_D$，即
$$
y = G(z) = W_D \cdot z + b_D = V \cdot z 
$$
其中$W_D = V, b_D = 0$

&emsp;&emsp;综上所述，当损失函数是平方损失函数时，自动编码器学习的目标函数是
$$
\begin{aligned}
L 
&= \frac{1}{N} \sum_{i=1}^N \| x_i - G(F(x_i)) \| \\
&= \frac{1}{N} \sum_{i=1}^N \| x_i - v_i \cdot (v_i^T \cdot x_i) \| \\
&= \frac{1}{N} \sum_{i=1}^N \| x_i - v_i v_i^T x_i \|
\end{aligned}
$$

&emsp;&emsp;由于单位特征向量$V$可以使得样本矩阵$X$的所有线性变换中方差最大，故单位向量特征$V$能够使目标函数最小，因此，当编码器和解码器都是线性函数，且$b_E = b_D = 0$时，主成分分析可以作为自动编码器的学习方法。

## 习题30.3

&emsp;&emsp;假设变分自编码器VAE模型的变分分布 $q_{\phi}(\boldsymbol{z} | \boldsymbol{x})$ 和似然函数 $p_{\boldsymbol{\theta}}(\boldsymbol{z}|\boldsymbol{x})$ 是多元高斯分布 $N(\boldsymbol{\mu}_e, \boldsymbol{\Sigma}_e)$ 和 $N(\boldsymbol{\mu}_d, \boldsymbol{\Sigma}_d)$。描述模型参数 $\phi, \theta$，高斯分布参数 $\boldsymbol{\mu}_e, \boldsymbol{\Sigma}_e, \boldsymbol{\mu}_d, \boldsymbol{\Sigma}_d$ 以及模型变量 $\boldsymbol{x}$ 和 $\boldsymbol{z}$ 之间的关系。

**解答：**

**解答思路：**

**解答步骤：**

## 习题30.4

&emsp;&emsp;推导式(30.19)的KL散度。

**解答：**

**解答思路：**

**解答步骤：**

## 习题30.5

&emsp;&emsp;证据下界有两种形式，式(30.14)和式(30.16)。30.3.3节中介绍的再参数化技巧是针对式(30.16)的。写出式(30.14)的证据下界的再参数化技巧的梯度公式。

**解答：**

**解答思路：**

**解答步骤：**

## 习题30.6

&emsp;&emsp;写出30.3.3节中的VAE的证据下界以及证据下界的梯度。

**解答：**

**解答思路：**

**解答步骤：**

## 习题30.7

&emsp;&emsp;比较30.3.3节中的VAE的证据下界和AE的学习目标。

**解答：**

**解答思路：**

**解答步骤：**

## 习题30.8

&emsp;&emsp;比较AE和VAE的编码器和解码器的神经网络。

**解答：**

**解答思路：**

**解答步骤：**

## 习题30.9

&emsp;&emsp;比较变分EM算法（第18章）和VAE学习算法。

**解答：**

**解答思路：**

**解答步骤：**